# GPT-2 355M Function Calling — Benchmark Evaluation

Loads the fine-tuned checkpoint from a Kaggle dataset input and evaluates on the Glaive Function Calling v2 test set.

**Before running:** Add your `gpt2-355m-function-calling` dataset as an input (right panel → Add Data).

In [ ]:
!pip install tiktoken huggingface-hub pandas -q

In [ ]:
%%writefile model.py
import math

import numpy as np
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by n_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.reshape(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    _coeff = math.sqrt(2.0 / math.pi)

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            self._coeff * (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_resid = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_resid(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_resid(x)
        x = x + shortcut

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


In [ ]:
%%writefile data.py
import torch
from torch.utils.data import Dataset


def format_entry(entry):
    system = entry['system'].strip().replace('SYSTEM:', '###SYSTEM:')

    turns = [
        t.strip()
         .replace("<|endoftext|>", "")
         .replace('USER:', '###USER:')
         .replace('ASSISTANT:', '###ASSISTANT:')
        for t in entry['chat'].split('\n\n\n')
        if t.strip()
    ]

    instruction_turns = []
    response = ""

    for turn in turns:
        if turn.startswith('###ASSISTANT:'):
            response = turn
            break
        instruction_turns.append(turn)

    instruction = system + '\n' + '\n'.join(instruction_turns)
    return instruction, response


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer, allowed_max_length=1024):
        self.data = data
        self.encoded_texts = []
        self.allowed_max_length = allowed_max_length

        for entry in data:
            prompt, response = format_entry(entry)
            if not response:
                continue

            prompt_ids = tokenizer.encode(prompt)
            response_ids = tokenizer.encode(response)
            token_ids = prompt_ids + response_ids

            if len(token_ids) + 1 > self.allowed_max_length:
                continue

            self.encoded_texts.append((token_ids, len(prompt_ids)))

    def __len__(self):
        return len(self.encoded_texts)

    def __getitem__(self, index):
        return self.encoded_texts[index]


def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(token_ids) + 1 for token_ids, _ in batch)
    if allowed_max_length is not None:
        batch_max_length = min(batch_max_length, allowed_max_length)

    inputs_lst, targets_lst = [], []

    for token_ids, prompt_len in batch:
        new_item = token_ids.copy()
        new_item += [pad_token_id]
        if allowed_max_length is not None:
            new_item = new_item[:allowed_max_length]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))

        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        targets[:prompt_len - 1] = ignore_index

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor


In [ ]:
%%writefile inference.py
import argparse
import json
import re

import pandas as pd
import tiktoken
import torch

from data import format_entry
from model import GPTModel


MODEL_CONFIGS = {
    "124M":  {"emb_dim": 768,  "n_layers": 12, "n_heads": 12},
    "355M":  {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "774M":  {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "1558M": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
}


def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, k=top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        if temperature > 0.0:
            logits = logits / temperature
            logits = logits - logits.max(dim=-1, keepdim=True).values
            probs = torch.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
        else:
            next_idx = torch.argmax(logits, dim=-1, keepdim=True)

        if next_idx == eos_id:
            break

        idx = torch.cat((idx, next_idx), dim=1)

    return idx


def extract_functioncall(text):
    """Parse `<functioncall> {...}` into a dict. Handles Glaive's single-quoted
    arguments blob: {"name": "f", "arguments": '{"k": "v"}'}."""
    start = text.find("<functioncall>")
    if start == -1:
        return None
    json_start = text.find("{", start)
    if json_start == -1:
        return None
    depth = 0
    span = None
    for i, ch in enumerate(text[json_start:], json_start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                span = text[json_start:i + 1]
                break
    if span is None:
        return None
    try:
        return json.loads(span)
    except json.JSONDecodeError:
        pass
    unwrapped = re.sub(r"'(\{.*\})'", r"\1", span, flags=re.DOTALL)
    try:
        return json.loads(unwrapped)
    except json.JSONDecodeError:
        return None


def run_inference(model, tokenizer, prompt, max_new_tokens, device):
    context_size = model.pos_emb.weight.shape[0]
    encoded = tokenizer.encode(prompt, allowed_special={"<|endoftext|>"})
    idx = torch.tensor(encoded).unsqueeze(0).to(device)
    out_idx = generate(model, idx, max_new_tokens=max_new_tokens, context_size=context_size, eos_id=50256)
    full_text = tokenizer.decode(out_idx.squeeze(0).tolist())
    return full_text[len(prompt):]


def compute_metrics(results):
    fc_results = [r for r in results if r["gt_fc"] is not None]
    total = len(results)
    fc_count = len(fc_results)

    if fc_count == 0:
        return {
            "total_samples": total,
            "fc_samples": 0,
            "non_fc_samples": total,
            "parse_rate": None,
            "fn_name_acc": None,
            "args_key_acc": None,
            "exact_match": None,
        }

    parse_rate = sum(1 for r in fc_results if r["pred_fc"] is not None) / fc_count
    parsed = [r for r in fc_results if r["pred_fc"] is not None]

    def keys_match(r):
        pred_args = r["pred_fc"].get("arguments", {})
        gt_args = r["gt_fc"].get("arguments", {})
        if not isinstance(pred_args, dict) or not isinstance(gt_args, dict):
            return False
        return set(pred_args.keys()) == set(gt_args.keys())

    fn_name_acc = sum(1 for r in parsed if r["pred_fc"].get("name") == r["gt_fc"].get("name")) / fc_count
    args_key_acc = sum(1 for r in parsed if keys_match(r)) / fc_count
    exact_match = sum(1 for r in parsed if r["pred_fc"] == r["gt_fc"]) / fc_count

    return {
        "total_samples": total,
        "fc_samples": fc_count,
        "non_fc_samples": total - fc_count,
        "parse_rate": round(parse_rate, 4),
        "fn_name_acc": round(fn_name_acc, 4),
        "args_key_acc": round(args_key_acc, 4),
        "exact_match": round(exact_match, 4),
    }


def main():
    parser = argparse.ArgumentParser(description="Evaluate fine-tuned GPT-2 on function calling")
    parser.add_argument("--checkpoint",          required=True)
    parser.add_argument("--model-size",          default="355M", choices=list(MODEL_CONFIGS))
    parser.add_argument("--num-samples",         type=int,   default=200, help="Max samples to evaluate (-1 = all)")
    parser.add_argument("--max-new-tokens",      type=int,   default=256)
    parser.add_argument("--holdout-start-frac",  type=float, default=0.85,
                        help="Fraction where holdout begins. Training used data[:85%], so 0.85 is safe.")
    parser.add_argument("--fc-only",             action="store_true",
                        help="Only evaluate entries where ground truth is a function call")
    parser.add_argument("--output-json",         default="results.json")
    args = parser.parse_args()

    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Device: {device}")

    cfg = {**BASE_CONFIG, **MODEL_CONFIGS[args.model_size]}
    model = GPTModel(cfg)
    state_dict = torch.load(args.checkpoint, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print(f"Loaded {args.model_size} checkpoint from {args.checkpoint}")

    tokenizer = tiktoken.get_encoding("gpt2")

    print("Loading Glaive dataset...")
    df = pd.read_json("hf://datasets/glaiveai/glaive-function-calling-v2/glaive-function-calling-v2.json")
    data = df.to_dict("records")
    holdout_start = int(len(data) * args.holdout_start_frac)
    holdout = data[holdout_start:]
    print(f"Holdout: data[{holdout_start}:] = {len(holdout)} entries (frac={args.holdout_start_frac})")
    if args.fc_only:
        print("Mode: fc-only (skipping entries where ground truth has no <functioncall>)")

    results = []
    evaluated = 0
    for entry in holdout:
        if args.num_samples > 0 and evaluated >= args.num_samples:
            break
        instruction, response = format_entry(entry)
        if not response:
            continue
        if args.fc_only and "<functioncall>" not in response:
            continue
        prediction = run_inference(model, tokenizer, instruction, args.max_new_tokens, device)
        gt_fc = extract_functioncall(response)
        pred_fc = extract_functioncall(prediction)
        results.append({
            "prompt": instruction,
            "ground_truth": response,
            "prediction": prediction,
            "gt_fc": gt_fc,
            "pred_fc": pred_fc,
        })
        evaluated += 1
        if evaluated % 50 == 0:
            print(f"  [{evaluated}]")

    metrics = compute_metrics(results)

    print("\n=== Results ===")
    for k, v in metrics.items():
        pct = f"  ({v * 100:.1f}%)" if isinstance(v, float) else ""
        print(f"  {k:<20} {v}{pct}")

    out = {"metrics": metrics, "samples": results}
    with open(args.output_json, "w") as f:
        json.dump(out, f, indent=2)
    print(f"\nSaved to {args.output_json}")


if __name__ == "__main__":
    main()


In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
CHECKPOINT_PATH = "/kaggle/input/gpt2-355m-function-calling/gpt2-355M-function-calling.pth"
MODEL_SIZE      = "355M"
NUM_SAMPLES     = -1     # -1 = full held-out set
MAX_NEW_TOKENS  = 256
OUTPUT_JSON     = "results.json"

# Use the full held-out range (last 15%) to avoid the dead-zone at the very end.
# Training used data[:85%], so data[85%:] is never seen by the model.
HOLDOUT_START_FRAC = 0.85

In [ ]:
import runpy, sys

sys.argv = [
    "inference.py",
    "--checkpoint",         CHECKPOINT_PATH,
    "--model-size",         MODEL_SIZE,
    "--num-samples",        str(NUM_SAMPLES),
    "--max-new-tokens",     str(MAX_NEW_TOKENS),
    "--holdout-start-frac", str(HOLDOUT_START_FRAC),
    "--fc-only",
    "--output-json",        OUTPUT_JSON,
]
runpy.run_path("inference.py", run_name="__main__")

In [ ]:
import runpy, sys

sys.argv = [
    "inference.py",
    "--checkpoint",     CHECKPOINT_PATH,
    "--model-size",     MODEL_SIZE,
    "--num-samples",    str(NUM_SAMPLES),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--output-json",    OUTPUT_JSON,
]
runpy.run_path("inference.py", run_name="__main__")

In [ ]:
import json
import pandas as pd

with open(OUTPUT_JSON) as f:
    results = json.load(f)

print("=== Metrics ===")
print(json.dumps(results["metrics"], indent=2))

print("\n=== Sample predictions (first 20) ===")
df = pd.DataFrame(results["samples"][:20])[["ground_truth", "prediction", "gt_fc", "pred_fc"]]
pd.set_option("display.max_colwidth", 120)
display(df)